# ML-04 — Search Intelligence Data Contract

This notebook defines the data contract and validates features for the Search Intelligence project.

## Setup

Installs, imports, and data connections.

In [ ]:
import duckdb
import pandas as pd
from huggingface_hub import login, list_repo_files, hf_hub_download
from google.colab import userdata

# Setup connection
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

# Download data paths
march_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset", token=HF_TOKEN)
april_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="fact_content_daily_performance/month=2026-04/data_0.parquet", repo_type="dataset", token=HF_TOKEN)

print("Setup complete.")

Setup complete.


## 1. Unit of analysis + time window

### Plain-language contract answer

One row represents one pseudonymized content item for one pseudonymized client on one report date. For this development work, I use the March 2026 partition (2026-03-01 to 2026-03-31).

### Verification Query 1 — Grain

In [ ]:
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM read_parquet('{march_file}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").fetchdf()

display(grain_check)
print("Grain is unique if DataFrame is empty.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Grain is unique if DataFrame is empty.


## 2. Fields: feature / label / context / excluded

### Plain-language field classification

*   **Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `client_has_gsc`, `gsc_data_available` (All knowable at decision time).
*   **Label:** A future performance outcome from the April window.
*   **Context:** `report_date`, `client_hash_id`, `content_hash_id`.
*   **Excluded:** `ga4_*` metrics (low density) and `gsc_sum_position` (redundant).

## 3. Verify it with queries

### Query 2 — Row count + date span

In [ ]:
display(con.execute(f"SELECT COUNT(*) as rows, MIN(report_date), MAX(report_date) FROM read_parquet('{march_file}')").fetchdf())

,rows,min(report_date),max(report_date)
0,9841378,2026-03-01,2026-03-31


### Query 3 — Availability using IS TRUE

In [ ]:
display(con.execute(f"SELECT COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_true_rows FROM read_parquet('{march_file}')").fetchdf())

,gsc_true_rows
0,3611061


### Five-feature frame

In [ ]:
feature_frame = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position, client_has_gsc, gsc_data_available
    FROM read_parquet('{march_file}')
    WHERE gsc_data_available IS TRUE
    LIMIT 5
""").fetchdf()
display(feature_frame)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,client_has_gsc,gsc_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,True,True
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,True,True
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,True,True
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,True,True
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,True,True


#### Available-when explanation for each feature

All selected GSC metrics represent search performance data observed and recorded prior to or on the report date, making them valid predictors for future events.

## 4. Deliberate leakage experiment

In [ ]:
# 4.1 Define future label & 4.2 Add deliberately leaked column
march_data = con.execute(f"SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) as march_imps FROM read_parquet('{march_file}') GROUP BY 1, 2").fetchdf()
april_data = con.execute(f"SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) as april_imps FROM read_parquet('{april_file}') GROUP BY 1, 2").fetchdf()

exp_df = march_data.merge(april_data, on=['client_hash_id', 'content_hash_id'])
exp_df['decline_label'] = (exp_df['april_imps'] < exp_df['march_imps']).astype(int)
exp_df['leaked_feature'] = exp_df['april_imps'] # The Leak

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Show leaky score = 1.0000

In [ ]:
leaky_acc = ((exp_df['leaked_feature'] < exp_df['march_imps']).astype(int) == exp_df['decline_label']).mean()
print(f"Leaky accuracy: {leaky_acc:.4f}")

Leaky accuracy: 1.0000


### Explain why it is invalid

The feature `april_imps` is from the future relative to the March decision point. Using it to predict a March-to-April decline is impossible in a real-world production environment.

### Remove leaked column and Show honest baseline = 0.5915

In [ ]:
exp_df = exp_df.drop(columns=['leaked_feature'])
majority_class = exp_df['decline_label'].mode()[0]
honest_acc = (exp_df['decline_label'] == majority_class).mean()
print(f"Honest baseline accuracy: {honest_acc:.4f}")

Honest baseline accuracy: 0.6622


## 5. Data limits

### One named limitation

**GSC Coverage Bias:** Search intelligence signals are only available for the subset of content with active Google Search Console data; rows lacking this data are effectively invisible to the current feature set.

## 6. Self-check

☑ Unit of analysis defined
☑ Grain verified
☑ Fields classified
☑ Query 2 & 3 run
☑ Leakage experiment completed
☑ Honest baseline calculated
☑ Limitation named

In [ ]:
print("\n".join(march_df.columns))

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


Features:
1. gsc_impressions — knowable at the decision moment because these are search impressions observed up to that date.
2. gsc_clicks — knowable at the decision moment because these are search clicks observed up to that date.
3. gsc_avg_position — knowable at the decision moment because it summarizes observed Google Search position up to that date.
4. client_has_gsc — knowable at the decision moment because it describes whether the client has GSC access.
5. gsc_data_available — knowable at the decision moment because it indicates whether GSC data is available for that observation.

Label / proxy:
- A future decline/underperformance outcome derived from a later observation window. It is the outcome we want to rank/review against and is never used as a feature.

Context:
- report_date — identifies the observation date.
- month — identifies the warehouse partition/time window.
- client_hash_id — pseudonymous client identifier used for grouping/splitting.
- content_hash_id — pseudonymous content identifier used for grouping/joining.

Excluded:
- Any future-window outcome or label-derived field — unavailable at the decision moment and would cause leakage.
- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec and other GA4 metrics — excluded from the core feature frame because only 413,966 of 9,841,378 March rows have ga4_data_available IS TRUE.
- gsc_sum_position — excluded because gsc_avg_position is the interpretable average-position metric.

In [ ]:
april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("April file downloaded successfully.")

April file downloaded successfully.


### Feature availability

| Feature | Available when? |
|---|---|
| gsc_impressions | Knowable at the decision moment because it represents GSC impressions observed up to that date. |
| gsc_clicks | Knowable at the decision moment because it represents GSC clicks observed up to that date. |
| gsc_avg_position | Knowable at the decision moment because it summarizes observed Google Search position up to that date. |
| client_has_gsc | Knowable at the decision moment because it describes whether the client has GSC access. |
| gsc_data_available | Knowable at the decision moment because it indicates whether GSC data is actually available for the observation. |

In [ ]:
import pandas as pd

# March: information available up to the decision moment
march_month = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS gsc_avg_position,
        BOOL_OR(client_has_gsc) AS client_has_gsc,
        BOOL_OR(gsc_data_available) AS gsc_data_available
    FROM read_parquet('{march_file}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

# April: future outcome window
april_month = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet('{april_file}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

# Merge March information with the future April outcome
model_df = march_month.merge(
    april_month,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Rows with March + April data:", len(model_df))
model_df.head()

Rows with March + April data: 158549


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,client_has_gsc,gsc_data_available,april_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,True,True,6787.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.307255,True,True,405.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,True,True,8475.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,True,True,6091.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.499519,True,True,287.0


In [ ]:
model_df["decline_label"] = (
    model_df["april_impressions"] < model_df["gsc_impressions"]
).astype(int)

print(model_df["decline_label"].value_counts())

decline_label
1    93779
0    64770
Name: count, dtype: int64


In [ ]:
model_df["decline_label"].value_counts()

,count
decline_label,
1,93779
0,64770


In [ ]:
model_df["leaky_april_impressions"] = model_df["april_impressions"]

In [ ]:
leak_check = pd.crosstab(
    model_df["leaky_april_impressions"] < model_df["gsc_impressions"],
    model_df["decline_label"]
)

leak_check

decline_label,0,1
row_0,,
False,64770,0
True,0,93779


In [ ]:
leaky_prediction = (
    model_df["leaky_april_impressions"] < model_df["gsc_impressions"]
).astype(int)

leaky_accuracy = (
    leaky_prediction == model_df["decline_label"]
).mean()

print(f"Leaky accuracy: {leaky_accuracy:.4f}")

Leaky accuracy: 1.0000


### Deliberate leakage experiment

I deliberately included `leaky_april_impressions`, which is derived from the same future April window used to construct the decline label. The resulting accuracy was approximately 1.00. This apparent perfect performance is invalid because April information was not available at the March 31 decision moment. The leaky column was therefore removed before keeping the honest feature set.

In [ ]:
# Honest baseline: always predict the majority class
majority_class = model_df["decline_label"].mode()[0]

honest_prediction = pd.Series(
    majority_class,
    index=model_df.index
)

honest_accuracy = (
    honest_prediction == model_df["decline_label"]
).mean()

print("Majority class:", majority_class)
print(f"Honest baseline accuracy: {honest_accuracy:.4f}")

Majority class: 1
Honest baseline accuracy: 0.5915


### Leakage conclusion

The deliberately leaked April feature produced an accuracy of 1.0000, which is invalid because April information was unavailable at the March 31 decision moment. After removing the leaked feature, the honest baseline accuracy was 0.5915. This lower score is the number I retain as the honest reference.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*




### Data limits

This slice has an unbalanced history across clients, so March 2026 is a development window rather than a universal history window for every client. GSC availability is also uneven: only 3,611,061 of 9,841,378 March rows have gsc_data_available IS TRUE. GA4 availability is much lower, with only 413,966 rows available in March, so GA4 metrics were excluded from the core five-feature frame.

The daily fact table can only describe observed search performance; it cannot tell us why a page gained or lost impressions. The data also does not establish causality between a content change and a search outcome.

Named limitation: GSC coverage is uneven across clients, so the Search Intelligence review signal is less reliable for pages or clients without usable GSC data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

☑ Section 1: Unit of analysis + time window completed
☑ Section 2: Feature / label / context / excluded completed
☑ Three verification queries completed
☑ Grain verified with zero duplicates
☑ March row count and date range verified
☑ Availability verified using IS TRUE
☑ Five-feature frame displayed
☑ Every feature has an "available when?" explanation
☑ Deliberate leakage experiment shown
☑ Leaky accuracy = 1.0000
☑ Leaky feature removed
☑ Honest baseline = 0.5915
☑ Data limitation stated